# 07 — NQS Comparison (NetKet)

> **spinq-vqe** | ARPA Quantum Logical Systems (QONDRA)

ED is exact but exponential. DMRG (NB06) is the polynomial MPS reference.
Reviewers also ask for a **Neural Quantum State** baseline: a neural-network
wavefunction optimized with Variational Monte Carlo (Carleo & Troyer 2017).

This notebook builds that baseline with [NetKet](https://www.netket.org/)
on the **same** Kagome-strip Hamiltonian as NB01–NB06, then places NQS beside
ED, DMRG, and VQE.

**Install:** `pip install -e ".[nqs]"`  
**Regenerate committed artifacts:** `python scripts/run_nqs_benchmark.py`

## What this establishes

| Check | Purpose |
|-------|---------|
| Hamiltonian match | NetKet reproduces the PennyLane ED matrix |
| Complex RBM | Recovers ED to ≪5% at N=9 (exact full-summation VMC) |
| RBMModPhase | Amplitude–phase factorization as a second architecture |
| Real RBM control | Shows why complex parameters are required on this strip |
| Four-way table | ED / DMRG / NQS / VQE on identical normalized energies |

### Scientific notes

- **Append-only figures:** this notebook writes `nqs_*.png` and
  `method_comparison.csv`. It does **not** overwrite DMRG or scaling figures.
- **Strip, not 2D Kagome:** we do not claim literature 2D Kagome GCNN accuracies.
  A strip-adapted GCNN underperformed here; RBMModPhase is the honest second model.
- **Exact vs sampled:** for N≤12 we use NetKet `FullSumState` (no MC noise).
  Sampled VMC is reserved for larger N.

### References
- Carleo & Troyer (2017) Science 355 — Neural Quantum States
- Astrakhantsev et al. (2021) PRX 11 — NQS on Kagome Heisenberg
- Vieijra et al. (2021) PRL 126 — symmetry-equivariant NQS


In [1]:
from __future__ import annotations
import os
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from spinq_vqe import nqs

os.makedirs('../figures', exist_ok=True)
os.makedirs('../data', exist_ok=True)

REPO = Path('..').resolve()
DATA = REPO / 'data'
FIG = REPO / 'figures'

ED_N9 = -1.4219039949999581
print('NetKet available:', nqs.NETKET_AVAILABLE)


NetKet available: True


---
## 1. Validate Hamiltonian against PennyLane ED

Same contract as TeNPy in NB06: one XX+YY+ZZ term per `kagome_graph` edge at
`J/(4N)`, plus the global `D/4` identity shift.


In [2]:
max_diff = nqs.validate_hamiltonian_against_pennylane(3)
print(f'max |H_netket - H_pennylane| at N=9: {max_diff:.3e}')
assert max_diff < 1e-10


max |H_netket - H_pennylane| at N=9: 0.000e+00


---
## 2. Why complex parameters matter

A real-valued RBM cannot encode the frustrated Kagome sign structure and
plateaus far above E₀. Complex RBM (or RBMModPhase) recovers the ground state.
This is a short diagnostic — full production runs live in the benchmark script.


In [3]:
# Short diagnostic (exact full-summation). Prefer the committed CSV for paper numbers.
real = nqs.run_nqs(
    3, model='rbm', n_iter=80, alpha=2.0, learning_rate=0.05,
    backend='exact', complex_params=False, optimizer='sgd', seed=42,
)
cplx = nqs.run_nqs(
    3, model='rbm', n_iter=300, alpha=2.0, learning_rate=0.01,
    backend='exact', complex_params=True, optimizer='adam', seed=42,
)

def err(e):
    return abs(e - ED_N9) / abs(ED_N9) * 100

print(f'Real RBM:     E0={real.e0:.6f}  err={err(real.e0):.2f}%')
print(f'Complex RBM:  E0={cplx.e0:.6f}  err={err(cplx.e0):.4f}%')
print(f'ED:           E0={ED_N9:.6f}')


Real RBM:     E0=-0.164959  err=88.40%
Complex RBM:  E0=-1.415127  err=0.4766%
ED:           E0=-1.421904


---
## 3. Method comparison table

Load committed `data/method_comparison.csv` (regenerate with the benchmark script).
VQE best energies come from NB02/NB05; DMRG from NB06.


In [4]:
csv_path = DATA / 'method_comparison.csv'
if not csv_path.exists():
    raise FileNotFoundError(
        'Missing data/method_comparison.csv — run: python scripts/run_nqs_benchmark.py'
    )

df = pd.read_csv(csv_path)
display_cols = [
    'n_sites', 'E_ED', 'E_DMRG', 'E_NQS_RBM', 'E_NQS_MODPHASE', 'E_VQE',
    'NQS_RBM_error_vs_ED_pct', 'NQS_MODPHASE_error_vs_ED_pct', 'VQE_error_vs_ED_pct',
]
df[display_cols]


 n_sites  n_cells      E_ED    E_DMRG  E_NQS_RBM  E_NQS_RBM_err  E_NQS_MODPHASE  E_NQS_MODPHASE_err     E_VQE  NQS_RBM_error_vs_ED_pct  NQS_MODPHASE_error_vs_ED_pct  VQE_error_vs_ED_pct NQS_backend
       9        3 -1.421904 -1.421904  -1.421831            0.0       -1.421812                 0.0 -1.284563                   0.0051                        0.0064               9.6589       exact
      12        4 -1.480418 -1.480418  -1.480155            0.0       -1.450947                 0.0 -1.238595                   0.0178                        1.9907              16.3348       exact


---
## 4. Committed figures (append-only)

These are generated by `scripts/run_nqs_benchmark.py` and are distinct from
NB06 DMRG figures.


In [5]:
from IPython.display import Image, display

for name in [
    'nqs_vmc_convergence.png',
    'nqs_method_comparison.png',
    'nqs_error_vs_ed.png',
]:
    path = FIG / name
    print(path.name, '→', 'OK' if path.exists() else 'MISSING')
    if path.exists():
        display(Image(filename=str(path), width=720))


nqs_vmc_convergence.png → OK
nqs_method_comparison.png → OK
nqs_error_vs_ed.png → OK


---
## 5. Takeaway for the paper

On this Kagome strip, a **complex RBM** (exact NQS) sits with ED/DMRG as a
classical variational reference, while VQE remains ~10–16% above E₀.
That gap is the quantum–classical story: NQS shows the Hamiltonian is
learnable with a flexible classical ansatz; VQE’s error is ansatz/optimizer
limited, not a missing classical baseline.

RBMModPhase provides an independent architecture check. Strip-adapted GCNN
was evaluated and underperformed with the small automorphism group of the
open strip — we report that negative result rather than overstating symmetry
claims.
